# 이벤트 기반 주식/뉴스 데이터 수집 노트북
## 누적 master 데이터 저장 전용

**목표**: 뉴스와 가격 데이터를 안전하게 누적 저장하고, 중복 없이 master CSV를 갱신

**권장 사용법**: 데이터 수집이 필요한 날에만 실행


# 빠른 사용 요약
1. 이 노트북은 데이터가 필요할 때만 실행합니다.
2. 뉴스와 가격 수집 결과는 `master CSV`에 누적 저장됩니다.
3. 수집이 끝나면 `2_ML_analysis.ipynb`에서 분석만 진행하면 됩니다.


# 1. 필수 라이브러리 설치 및 임포트

In [1]:
import os
import re
import json
import warnings
from pathlib import Path
from datetime import datetime, timedelta, timezone
import time
import random

import numpy as np
import pandas as pd
import requests
import yfinance as yf
import hashlib

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

warnings.filterwarnings("ignore")
rcParams["font.family"] = "DejaVu Sans"
sns.set_style("whitegrid")

print("Imports loaded")
print("GNEWS_API_KEY loaded:", bool(os.getenv("GNEWS_API_KEY", "").strip()))
print("NEWSAPI_KEY loaded:", bool(os.getenv("NEWSAPI_KEY", "").strip()))


Imports loaded
GNEWS_API_KEY loaded: True
NEWSAPI_KEY loaded: False


# 2. 사용자 수동 설정

- 이 셀에서만 실제 수집 플래그와 핵심 변수를 바꾸세요.
- 이 노트북은 수집 전용이라 수집 플래그만 조절하면 됩니다.
- API 호출을 원하는 날에만 `True`로 바꾸고 실행하세요.


In [2]:
USER_ENABLE_NEWS_API_FETCH = True
USER_ENABLE_PRICE_REFRESH = True
USER_LOOKBACK_DAYS = 730
USER_NEWS_COLLECTION_LOOKBACK_DAYS = 30
USER_FORECAST_HORIZONS = [5, 20, 60]
USER_PRIMARY_FORECAST_HORIZON = 20
USER_MIN_TRAIN_ROWS = 120
USER_MIN_NEWS_HISTORY_DAYS = 45
USER_MAX_API_ARTICLES_PER_QUERY = 100
USER_GNEWS_MIN_REQUEST_INTERVAL_SECONDS = 2.5
USER_GNEWS_MAX_RETRIES = 4
USER_GNEWS_BACKOFF_BASE_SECONDS = 4.0

print("Collection control panel")
print(f"- USER_ENABLE_NEWS_API_FETCH: {USER_ENABLE_NEWS_API_FETCH}")
print(f"- USER_ENABLE_PRICE_REFRESH: {USER_ENABLE_PRICE_REFRESH}")
print(f"- USER_LOOKBACK_DAYS: {USER_LOOKBACK_DAYS}")
print(f"- USER_NEWS_COLLECTION_LOOKBACK_DAYS: {USER_NEWS_COLLECTION_LOOKBACK_DAYS}")
print(f"- USER_FORECAST_HORIZONS: {USER_FORECAST_HORIZONS}")
print(f"- USER_PRIMARY_FORECAST_HORIZON: {USER_PRIMARY_FORECAST_HORIZON}")
print(f"- USER_MAX_API_ARTICLES_PER_QUERY: {USER_MAX_API_ARTICLES_PER_QUERY}")
print(f"- USER_GNEWS_MIN_REQUEST_INTERVAL_SECONDS: {USER_GNEWS_MIN_REQUEST_INTERVAL_SECONDS}")
print(f"- USER_GNEWS_MAX_RETRIES: {USER_GNEWS_MAX_RETRIES}")
print(f"- USER_GNEWS_BACKOFF_BASE_SECONDS: {USER_GNEWS_BACKOFF_BASE_SECONDS}")


Collection control panel
- USER_ENABLE_NEWS_API_FETCH: True
- USER_ENABLE_PRICE_REFRESH: True
- USER_LOOKBACK_DAYS: 730
- USER_NEWS_COLLECTION_LOOKBACK_DAYS: 30
- USER_FORECAST_HORIZONS: [5, 20, 60]
- USER_PRIMARY_FORECAST_HORIZON: 20
- USER_MAX_API_ARTICLES_PER_QUERY: 100
- USER_GNEWS_MIN_REQUEST_INTERVAL_SECONDS: 2.5
- USER_GNEWS_MAX_RETRIES: 4
- USER_GNEWS_BACKOFF_BASE_SECONDS: 4.0


# 3. API 설정 및 수집 대상 정의

- 뉴스 API, 가격 데이터, 추적할 테마/자산군을 설정합니다.


In [3]:
ANALYSIS_DATE = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)
LOOKBACK_DAYS = int(globals().get("USER_LOOKBACK_DAYS", 730))
FORECAST_HORIZONS = list(globals().get("USER_FORECAST_HORIZONS", [5, 20, 60]))
NEWS_COLLECTION_LOOKBACK_DAYS = int(globals().get("USER_NEWS_COLLECTION_LOOKBACK_DAYS", 30))
PRIMARY_FORECAST_HORIZON = int(globals().get("USER_PRIMARY_FORECAST_HORIZON", 20))
MIN_TRAIN_ROWS = int(globals().get("USER_MIN_TRAIN_ROWS", 120))
MIN_NEWS_HISTORY_DAYS = int(globals().get("USER_MIN_NEWS_HISTORY_DAYS", 45))
ENABLE_NEWS_API_FETCH = bool(globals().get("USER_ENABLE_NEWS_API_FETCH", False))
ENABLE_PRICE_REFRESH = bool(globals().get("USER_ENABLE_PRICE_REFRESH", False))
ALLOW_WEAK_MODEL_RECOMMENDATIONS = False
MAX_API_ARTICLES_PER_QUERY = int(globals().get("USER_MAX_API_ARTICLES_PER_QUERY", 100))
GNEWS_FREE_MAX_ARTICLES = 10
MAX_NEWS_API_REQUESTS = 100
GNEWS_MIN_REQUEST_INTERVAL_SECONDS = float(globals().get("USER_GNEWS_MIN_REQUEST_INTERVAL_SECONDS", 2.5))
GNEWS_MAX_RETRIES = int(globals().get("USER_GNEWS_MAX_RETRIES", 4))
GNEWS_BACKOFF_BASE_SECONDS = float(globals().get("USER_GNEWS_BACKOFF_BASE_SECONDS", 4.0))

# 저장 루트
WORKSPACE_ROOT = Path.cwd()
if (WORKSPACE_ROOT / "1_data_collection.ipynb").exists() or (WORKSPACE_ROOT / "2_ML_analysis.ipynb").exists():
    STOCK_ANALYSIS_DIR = WORKSPACE_ROOT
elif (WORKSPACE_ROOT / "Stock_analysis" / "1_data_collection.ipynb").exists() or (WORKSPACE_ROOT / "Stock_analysis" / "2_ML_analysis.ipynb").exists():
    STOCK_ANALYSIS_DIR = WORKSPACE_ROOT / "Stock_analysis"
else:
    STOCK_ANALYSIS_DIR = WORKSPACE_ROOT / "Stock_analysis"

# 저장 루트
DATA_DIR = STOCK_ANALYSIS_DIR / "data" / "event_driven_stock_predictor"
DATA_DIR.mkdir(parents=True, exist_ok=True)

NEWS_SNAPSHOT_DIR = DATA_DIR / "news_snapshots"
PRICE_SNAPSHOT_DIR = DATA_DIR / "price_snapshots"
NEWS_SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
PRICE_SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

NEWS_MASTER_PATH = DATA_DIR / "news_master_history.csv"
PRICE_MASTER_PATH = DATA_DIR / "price_master_history.csv"

RUN_TS = datetime.now(timezone.utc)

NEWS_COLUMNS = ["date", "keyword", "title", "description", "source", "url", "risk_score", "article_id", "title_key"]
PRICE_COLUMNS = [
    "date", "close", "volume", "symbol", "asset_name", "category", "kind",
    "return_1d", "vol_20d", "mom_5d", "mom_20d", "volume_chg_5d"
]

def _latest_snapshot_path(snapshot_dir: Path, prefix: str):
    files = sorted(snapshot_dir.glob(f"{prefix}_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
    return files[0] if files else None

def _df_hash(df: pd.DataFrame) -> str:
    if df is None:
        return ""
    d = df.copy()
    d = d.reindex(sorted(d.columns), axis=1)
    payload = d.to_csv(index=False).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def save_snapshot_if_changed(df: pd.DataFrame, snapshot_dir: Path, prefix: str, run_ts: datetime):
    latest = _latest_snapshot_path(snapshot_dir, prefix)
    if df is None or len(df) == 0:
        return latest, False
    if latest is not None:
        try:
            old_df = pd.read_csv(latest)
            if _df_hash(old_df) == _df_hash(df):
                return latest, False
        except Exception:
            pass
    out_path = snapshot_dir / f"{prefix}_{run_ts:%Y%m%d_%H%M%S}.csv"
    df.to_csv(out_path, index=False, encoding="utf-8")
    return out_path, True

def load_latest_snapshot_df(snapshot_dir: Path, prefix: str, empty_columns=None):
    latest = _latest_snapshot_path(snapshot_dir, prefix)
    if latest is None:
        return pd.DataFrame(columns=empty_columns or [])
    return pd.read_csv(latest)

def load_history_csv(path: Path, empty_columns=None):
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame(columns=empty_columns or [])

def load_env_file(env_path: Path):
    values = {}
    if not env_path.exists():
        return values
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key:
            values[key] = value
    return values

def _looks_unusable_api_key(value: str) -> bool:
    if not value:
        return False
    lowered = value.lower()
    placeholders = [
        "actual_api_key",
        "your_api_key",
        "replace_me",
        "placeholder_key",
        "sample_key",
    ]
    if any(token in lowered for token in placeholders):
        return True
    if any(ord(ch) > 127 for ch in value):
        return True
    return False

def resolve_api_key(name: str) -> str:
    env_value = os.getenv(name, "").strip()
    if env_value and not _looks_unusable_api_key(env_value):
        return env_value
    candidates = [
        STOCK_ANALYSIS_DIR / ".env",
        STOCK_ANALYSIS_DIR.parent / ".env",
    ]
    for env_path in candidates:
        values = load_env_file(env_path)
        value = values.get(name, "").strip()
        if value and not _looks_unusable_api_key(value):
            return value
    return ""

def looks_like_placeholder_key(value: str) -> bool:
    if not value:
        return False
    lowered = value.lower()
    return _looks_unusable_api_key(value)

def _normalize_url(value):
    raw = str(value).strip()
    if not raw or raw.lower() == "nan":
        return ""
    return re.sub(r"[?#].*$", "", raw.lower())

def _normalize_title(value):
    text = str(value or "").strip().lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def _article_id_from_row(row):
    normalized_url = _normalize_url(row.get("url", ""))
    if normalized_url:
        return hashlib.sha256(normalized_url.encode("utf-8")).hexdigest()
    title_key = _normalize_title(row.get("title", ""))
    source = str(row.get("source", "") or "").strip().lower()
    keyword = str(row.get("keyword", "") or "").strip().lower()
    date_part = pd.to_datetime(row.get("date"), utc=True, errors="coerce")
    date_part = date_part.strftime("%Y-%m-%d") if pd.notna(date_part) else ""
    payload = f"{title_key}|{source}|{keyword}|{date_part}"
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def prepare_news_history(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=NEWS_COLUMNS)
    base = df.copy()
    for c in NEWS_COLUMNS:
        if c not in base.columns:
            base[c] = np.nan
    base["date"] = pd.to_datetime(base["date"], utc=True, errors="coerce")
    base = base.dropna(subset=["date"])
    for c in ["keyword", "title", "description", "source", "url"]:
        base[c] = base[c].fillna("").astype(str).str.strip()
    base["risk_score"] = pd.to_numeric(base["risk_score"], errors="coerce").fillna(0.0).clip(0, 1)
    base["title_key"] = base["title"].map(_normalize_title)
    base["article_id"] = base.apply(_article_id_from_row, axis=1)
    base["publish_day"] = base["date"].dt.floor("D")
    base = base.sort_values(["date", "article_id"])
    base = base.drop_duplicates(subset=["article_id"], keep="last")
    base = base.drop_duplicates(subset=["title_key", "source", "publish_day"], keep="last")
    return base[NEWS_COLUMNS].reset_index(drop=True)

def prepare_price_history(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=PRICE_COLUMNS)
    base = df.copy()
    for c in PRICE_COLUMNS:
        if c not in base.columns:
            base[c] = np.nan
    base["date"] = pd.to_datetime(base["date"], utc=True, errors="coerce")
    numeric_cols = ["close", "volume", "return_1d", "vol_20d", "mom_5d", "mom_20d", "volume_chg_5d"]
    for c in numeric_cols:
        base[c] = pd.to_numeric(base[c], errors="coerce")
    base = base.dropna(subset=["date", "symbol", "close"])
    base = base.sort_values(["symbol", "date"]).drop_duplicates(subset=["date", "symbol"], keep="last")
    return base[PRICE_COLUMNS].reset_index(drop=True)

def merge_history(existing_df: pd.DataFrame, incoming_df: pd.DataFrame, prepare_fn, sort_cols, dedupe_cols):
    frames = []
    if existing_df is not None and len(existing_df) > 0:
        frames.append(existing_df)
    if incoming_df is not None and len(incoming_df) > 0:
        frames.append(incoming_df)
    if not frames:
        return prepare_fn(pd.DataFrame())
    merged = pd.concat(frames, ignore_index=True)
    merged = prepare_fn(merged)
    merged = merged.sort_values(sort_cols).drop_duplicates(subset=dedupe_cols, keep="last")
    return merged.reset_index(drop=True)

def upsert_history_file(path: Path, incoming_df: pd.DataFrame, prepare_fn, sort_cols, dedupe_cols):
    existing_df = load_history_csv(path)
    merged = merge_history(existing_df, incoming_df, prepare_fn, sort_cols, dedupe_cols)
    changed = _df_hash(existing_df) != _df_hash(merged)
    if changed and len(merged) > 0:
        merged.to_csv(path, index=False, encoding="utf-8")
    return merged, changed

COMMODITIES = {}

STOCKS = {
    "Market_Benchmark": ["SPY", "QQQ"],
    "Semiconductor": ["SOXX", "NVDA"],
    "Defense_Aerospace": ["ITA", "LMT"],
    "Airlines": ["JETS", "DAL"],
    "Oil_Energy": ["XLE", "XOM"],
    "Coal": ["BTU"],
    "LNG_Gas": ["LNG", "EQT"],
    "Shipbuilding": ["HII"],
    "Industrials": ["XLI", "GE"],
    "Trade_Logistics": ["IYT", "UPS"],
    "Banks": ["XLF", "JPM"],
    "Gold_Miners": ["GDX", "NEM"],
    "Rare_Earths": ["MP"],
    "Lithium": ["ALB"],
}

CATEGORY_LABELS = {
    "Market_Benchmark": "미국 대표지수",
    "Semiconductor": "반도체",
    "Defense_Aerospace": "방산/항공우주",
    "Airlines": "항공",
    "Oil_Energy": "석유/에너지",
    "Coal": "석탄",
    "LNG_Gas": "LNG/천연가스",
    "Shipbuilding": "조선",
    "Industrials": "제조/산업재",
    "Trade_Logistics": "물류/운송",
    "Banks": "은행",
    "Gold_Miners": "금 관련주",
    "Rare_Earths": "희토류",
    "Lithium": "리튬",
}

ASSET_LABELS = {
    "SPY": "미국 대형주 ETF(S&P 500)",
    "QQQ": "미국 기술주 ETF(나스닥 100)",
    "SOXX": "반도체 ETF",
    "NVDA": "엔비디아(AI 반도체)",
    "ITA": "미국 방산 ETF",
    "LMT": "록히드마틴(방산)",
    "JETS": "항공 ETF",
    "DAL": "델타항공",
    "XLE": "에너지 ETF",
    "XOM": "엑슨모빌",
    "BTU": "피바디 에너지(석탄)",
    "LNG": "셰니에르 에너지(LNG)",
    "EQT": "EQT(천연가스 생산)",
    "HII": "헌팅턴 잉걸스(조선)",
    "XLI": "산업재 ETF",
    "GE": "GE 에어로스페이스",
    "IYT": "미국 운송 ETF",
    "UPS": "UPS(물류)",
    "XLF": "금융 ETF",
    "JPM": "JP모건",
    "GDX": "금광주 ETF",
    "NEM": "뉴몬트(금광)",
    "MP": "MP 머티리얼즈(희토류)",
    "ALB": "앨버말(리튬)",
}

CURRENT_PRICE_SYMBOLS = set(COMMODITIES.values()) | {s for syms in STOCKS.values() for s in syms}

EVENT_TOPICS = {
    "Geopolitical_Conflict": [  # 지정학 분쟁/전쟁/제재 리스크
        "red sea shipping",  # 홍해 해상 운송 차질
        "shipping disruption",  # 해운 차질
        "oil supply disruption",  # 원유 공급 차질
        "sanctions risk",  # 제재 리스크
        "middle east conflict",  # 중동 분쟁
        "tanker attacks",  # 유조선 공격
        "russia ukraine energy",  # 러-우 전쟁발 에너지 이슈
        "defense spending",  # 국방비 지출 확대
    ],
    "Monetary_Inflation": [  # 금리/물가/고용/달러
        "fed rate cut",  # 연준 금리 인하
        "fed rate hike",  # 연준 금리 인상
        "treasury yields",  # 미 국채 금리
        "yield curve inversion",  # 장단기 금리 역전
        "us cpi",  # 미국 소비자물가
        "pce inflation",  # PCE 물가
        "nonfarm payrolls",  # 미국 비농업 고용
        "labor market slowdown",  # 고용시장 둔화
        "dollar strength",  # 달러 강세
    ],
    "Tariff_Trade_Policy": [  # 관세/무역정책/수출입 규제
        "tariff hike",  # 관세 인상
        "tariff inflation",  # 관세발 물가 압력
        "trade war",  # 무역 전쟁
        "export controls",  # 수출 통제
        "import duties",  # 수입 관세
        "us china trade",  # 미중 무역 갈등
        "supply chain reshoring",  # 공급망 리쇼어링
    ],
    "China_Demand_Stimulus": [  # 중국 경기/부양책/원자재 수요
        "china stimulus",  # 중국 경기 부양책
        "china property crisis",  # 중국 부동산 위기
        "china pmi",  # 중국 PMI
        "china manufacturing",  # 중국 제조업 경기
        "copper demand china",  # 중국 구리 수요
        "steel demand china",  # 중국 철강 수요
        "china demand recovery",  # 중국 수요 회복
        "yuan weakness",  # 위안화 약세
    ],
    "Energy_Oil_Gas": [  # 유가/정제마진/LNG/천연가스
        "opec+ production",  # OPEC+ 생산 정책
        "crude oil inventory",  # 원유 재고
        "oil prices",  # 유가
        "brent crude",  # 브렌트유
        "oil refinery outage",  # 정유 설비 가동 중단
        "diesel crack spread",  # 디젤 크랙 스프레드
        "lng supply",  # LNG 공급
        "natural gas storage",  # 천연가스 재고
    ],
    "Shipping_Logistics": [  # 해운 운임/물류 차질/항만 병목
        "container shipping rates",  # 컨테이너 운임
        "freight rates",  # 화물 운임
        "port congestion",  # 항만 적체
        "ocean freight rates",  # 해상 운임
        "logistics disruption",  # 물류 차질
        "shipping bottlenecks",  # 해운 병목
        "air cargo demand",  # 항공 화물 수요
    ],
    "Semiconductor_AI": [  # 반도체 사이클/AI 투자/메모리 수급
        "ai chip demand",  # AI 반도체 수요
        "gpu supply",  # GPU 공급
        "data center capex",  # 데이터센터 투자
        "hyperscaler capex",  # 빅테크 인프라 투자
        "hbm demand",  # HBM 수요
        "foundry utilization",  # 파운드리 가동률
        "semiconductor inventory",  # 반도체 재고
        "memory pricing",  # 메모리 가격
        "server supply chain",  # 서버 공급망
        "tsmc capacity",  # TSMC 생산능력
        "semiconductor export controls",  # 반도체 수출 규제
    ],
    "Defense_Aerospace": [  # 국방예산/방산 수주/조선
        "defense budget",  # 국방 예산
        "defense contract",  # 방산 계약 수주
        "munitions demand",  # 탄약 수요
        "missile defense",  # 미사일 방어
        "fighter jet order",  # 전투기 발주
        "military spending",  # 군사비 지출
        "nato spending",  # NATO 국방비
        "shipbuilding backlog",  # 조선 수주 잔고
    ],
    "Financial_Stress": [  # 신용스프레드/유동성/금융불안
        "credit spreads",  # 신용 스프레드
        "high yield spreads",  # 하이일드 스프레드
        "high yield default",  # 투기등급 채무불이행
        "commercial real estate stress",  # 상업용 부동산 부실
        "bank liquidity stress",  # 은행 유동성 불안
        "regional bank stress",  # 지방은행 불안
        "private credit stress",  # 사모신용 불안
        "funding market stress",  # 자금시장 경색
        "liquidity crunch",  # 유동성 위축
        "distressed debt",  # 부실채권 확대
        "debt ceiling standoff",  # 미국 부채한도 대치
    ],
    "Precious_Metals_Safe_Haven": [  # 금 가격/안전자산 선호
        "gold prices",  # 금 가격
        "gold safe haven",  # 금 안전자산 수요
        "central bank gold buying",  # 중앙은행 금 매입
        "bullion demand",  # 실물 금 수요
        "real yields gold",  # 실질금리와 금 가격
        "safe haven demand",  # 안전자산 선호
    ],
    "Battery_Materials": [  # 리튬/희토류/배터리 소재
        "lithium prices",  # 리튬 가격
        "lithium oversupply",  # 리튬 공급 과잉
        "battery metals",  # 배터리 금속
        "ev demand slowdown",  # 전기차 수요 둔화
        "rare earth supply",  # 희토류 공급
        "critical minerals",  # 핵심 광물
        "battery storage demand",  # 에너지저장장치 수요
    ],
    "Earnings_Guidance": [  # 실적/가이던스/마진/수요둔화
        "earnings beat",  # 실적 상회
        "earnings miss",  # 실적 하회
        "revenue guidance",  # 매출 가이던스
        "profit warning",  # 실적 경고
        "margin pressure",  # 마진 압박
        "demand slowdown",  # 수요 둔화
        "capex guidance",  # 설비투자 가이던스
        "guidance cut",  # 가이던스 하향
        "guidance raise",  # 가이던스 상향
    ],
}

EVENT_KEYWORDS = list(dict.fromkeys([kw for kws in EVENT_TOPICS.values() for kw in kws]))
if len(EVENT_KEYWORDS) >= MAX_NEWS_API_REQUESTS:
    raise ValueError(
        f"EVENT_KEYWORDS count ({len(EVENT_KEYWORDS)}) must stay below MAX_NEWS_API_REQUESTS ({MAX_NEWS_API_REQUESTS}). "
        "Reduce topic keywords before running live news collection."
    )

POSITIVE_RISK_WORDS = [
    "war", "conflict", "crisis", "attack", "explosion", "sanctions", "embargo", "blockade",
    "disruption", "shortage", "recession", "default", "bank stress", "liquidity", "downgrade",
]

print(f"Analysis date: {ANALYSIS_DATE:%Y-%m-%d}")
print(f"Lookback days: {LOOKBACK_DAYS}")
print(f"Forecast horizons: {FORECAST_HORIZONS} (primary={PRIMARY_FORECAST_HORIZON}d)")
print(f"News API fetch enabled: {ENABLE_NEWS_API_FETCH}")
print(f"Price refresh enabled: {ENABLE_PRICE_REFRESH}")
print(f"Master news path: {NEWS_MASTER_PATH}")
print(f"Master price path: {PRICE_MASTER_PATH}")
print(f"Commodities: {len(COMMODITIES)}")
print(f"Stock categories: {len(STOCKS)}, symbols: {sum(len(v) for v in STOCKS.values())}")
print(f"Event topics: {len(EVENT_TOPICS)}, query keywords: {len(EVENT_KEYWORDS)}")
print(f"Data dir: {DATA_DIR.resolve()}")


Analysis date: 2026-03-27
Lookback days: 730
Forecast horizons: [5, 20, 60] (primary=20d)
News API fetch enabled: True
Price refresh enabled: True
Master news path: c:\Users\user\Documents\GitHub\GB_practicing\Stock_analysis\data\event_driven_stock_predictor\news_master_history.csv
Master price path: c:\Users\user\Documents\GitHub\GB_practicing\Stock_analysis\data\event_driven_stock_predictor\price_master_history.csv
Commodities: 0
Stock categories: 14, symbols: 24
Event topics: 12, query keywords: 99
Data dir: C:\Users\user\Documents\GitHub\GB_practicing\Stock_analysis\data\event_driven_stock_predictor


# 4. 수집 실행 안내

- `ENABLE_NEWS_API_FETCH = True`를 켜야 실제 뉴스 API를 호출합니다.
- `ENABLE_PRICE_REFRESH = True`를 켜야 가격 데이터를 새로 갱신합니다.
- 두 플래그가 모두 `False`면 실제 수집 없이 `master CSV`만 점검합니다.


In [4]:
REQUESTED_NEWS_API_FETCH = bool(ENABLE_NEWS_API_FETCH)
REQUESTED_PRICE_REFRESH = bool(ENABLE_PRICE_REFRESH)

print("Collection guide")
print(f"- REQUESTED_NEWS_API_FETCH: {REQUESTED_NEWS_API_FETCH}")
print(f"- REQUESTED_PRICE_REFRESH: {REQUESTED_PRICE_REFRESH}")

if not (REQUESTED_NEWS_API_FETCH or REQUESTED_PRICE_REFRESH):
    print("\n[WARNING] Both live collection flags are OFF.")
    print("[WARNING] This run will NOT call the news API or refresh prices.")
    print("[WARNING] Turn on ENABLE_NEWS_API_FETCH and/or ENABLE_PRICE_REFRESH before collecting.")
else:
    print("\nCollection notebook is ready. Live collection will run only for the flags set to True.")


Collection guide
- REQUESTED_NEWS_API_FETCH: True
- REQUESTED_PRICE_REFRESH: True

Collection notebook is ready. Live collection will run only for the flags set to True.


# 5. 뉴스 및 이벤트 데이터 수집


In [5]:
def _safe_text(value):
    return str(value) if value is not None else ""

def _risk_score_from_text(title, description):
    text = f"{_safe_text(title)} {_safe_text(description)}".lower()
    hits = sum(1 for w in POSITIVE_RISK_WORDS if w in text)
    return min(hits / max(len(POSITIVE_RISK_WORDS), 1), 1.0)

def fetch_news_from_api(keywords, from_dt, to_dt, enabled=False):
    if not enabled:
        print("ENABLE_NEWS_API_FETCH=False -> skipping live news API requests.")
        return pd.DataFrame(columns=NEWS_COLUMNS)
    gnews_key = resolve_api_key("GNEWS_API_KEY")
    newsapi_key = resolve_api_key("NEWSAPI_KEY")
    all_rows = []
    print(f"API key status -> GNEWS: {bool(gnews_key)} (len={len(gnews_key)}), NEWSAPI: {bool(newsapi_key)} (len={len(newsapi_key)})")
    if looks_like_placeholder_key(gnews_key):
        print("[WARNING] GNEWS_API_KEY appears to be a placeholder value. GNews will be skipped.")
        print("[WARNING] Put the real key in your environment variables or in Stock_analysis/.env if you want GNews.")
        gnews_key = ""
    if gnews_key and len(gnews_key) < 16:
        print("[WARNING] GNEWS_API_KEY length looks unusually short. Please verify the key value.")
    if looks_like_placeholder_key(newsapi_key):
        print("[WARNING] NEWSAPI_KEY appears to be a placeholder value. NewsAPI will be skipped.")
        newsapi_key = ""
    if not gnews_key and not newsapi_key:
        print("No GNEWS_API_KEY or NEWSAPI_KEY found. API collection skipped.")
        return pd.DataFrame(columns=NEWS_COLUMNS)
    gnews_from_dt = max(from_dt, to_dt - timedelta(days=NEWS_COLLECTION_LOOKBACK_DAYS))
    gnews_max_articles = min(MAX_API_ARTICLES_PER_QUERY, GNEWS_FREE_MAX_ARTICLES)
    print(f"Requested news window: {from_dt:%Y-%m-%d} -> {to_dt:%Y-%m-%d}")
    print(f"Effective GNews window: {gnews_from_dt:%Y-%m-%d} -> {to_dt:%Y-%m-%d} (free-plan safe)")
    print(f"Effective GNews max articles/query: {gnews_max_articles}")

    gnews_last_request_ts = 0.0
    gnews_rate_limited_until = 0.0

    def _normalize_gnews_query(keyword):
        return re.sub(r"\s+", " ", str(keyword).replace("+", " ")).strip()

    def query_gnews(keyword):
        nonlocal gnews_last_request_ts, gnews_rate_limited_until
        if not gnews_key:
            return []
        url = "https://gnews.io/api/v4/search"
        safe_keyword = _normalize_gnews_query(keyword)
        params = {
            "q": safe_keyword,
            "lang": "en",
            "from": gnews_from_dt.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "to": to_dt.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "sortby": "publishedAt",
            "max": gnews_max_articles,
            "apikey": gnews_key,
        }
        for attempt in range(1, GNEWS_MAX_RETRIES + 1):
            now_ts = time.time()
            wait_seconds = max(0.0, gnews_last_request_ts + GNEWS_MIN_REQUEST_INTERVAL_SECONDS - now_ts, gnews_rate_limited_until - now_ts)
            if wait_seconds > 0:
                sleep_for = wait_seconds + random.uniform(0.1, 0.6)
                print(f"GNews throttle: waiting {sleep_for:.1f}s before keyword={keyword!r}")
                time.sleep(sleep_for)
            try:
                r = requests.get(url, params=params, timeout=15)
                gnews_last_request_ts = time.time()
                if r.status_code == 200:
                    payload = r.json()
                    articles = payload.get("articles", [])
                    total = payload.get("totalArticles")
                    print(f"GNews keyword={keyword!r} -> totalArticles={total}, fetched={len(articles)}")
                    return articles
                body = r.text[:300] if getattr(r, "text", None) else ""
                if r.status_code == 429:
                    retry_after = r.headers.get("Retry-After", "")
                    retry_after_seconds = float(retry_after) if str(retry_after).strip().isdigit() else 0.0
                    backoff_seconds = max(retry_after_seconds, GNEWS_BACKOFF_BASE_SECONDS * (2 ** (attempt - 1)))
                    gnews_rate_limited_until = time.time() + backoff_seconds
                    print(f"GNews rate limit hit: keyword={keyword!r}, attempt={attempt}/{GNEWS_MAX_RETRIES}, backoff={backoff_seconds:.1f}s")
                    if attempt < GNEWS_MAX_RETRIES:
                        continue
                print(f"GNews error: status={r.status_code}, keyword={keyword}, query={safe_keyword}, body={body}")
                break
            except Exception as e:
                print(f"GNews request error (keyword={keyword}, attempt={attempt}/{GNEWS_MAX_RETRIES}): {e}")
                if attempt < GNEWS_MAX_RETRIES:
                    time.sleep(min(GNEWS_BACKOFF_BASE_SECONDS * attempt, 15.0))
                    continue
                break
        return []

    def query_newsapi(keyword):
        if not newsapi_key:
            return []
        url = "https://newsapi.org/v2/everything"
        params = {
            "q": keyword,
            "language": "en",
            "from": from_dt.strftime("%Y-%m-%d"),
            "to": to_dt.strftime("%Y-%m-%d"),
            "sortBy": "publishedAt",
            "pageSize": MAX_API_ARTICLES_PER_QUERY,
            "apiKey": newsapi_key,
        }
        try:
            r = requests.get(url, params=params, timeout=15)
            if r.status_code == 200:
                payload = r.json()
                articles = payload.get("articles", [])
                print(f"NewsAPI keyword={keyword!r} -> fetched={len(articles)}")
                return articles
            body = r.text[:300] if getattr(r, "text", None) else ""
            print(f"NewsAPI error: status={r.status_code}, keyword={keyword}, body={body}")
        except Exception as e:
            print(f"NewsAPI request error (keyword={keyword}): {e}")
        return []

    for kw in keywords:
        articles = query_gnews(kw)
        if not articles:
            articles = query_newsapi(kw)
        for a in articles:
            date_raw = a.get("publishedAt") or a.get("published_at")
            if not date_raw:
                continue
            try:
                dt = pd.to_datetime(date_raw, utc=True)
            except Exception:
                continue
            title = _safe_text(a.get("title"))
            desc = _safe_text(a.get("description"))
            src = a.get("source", "Unknown")
            if isinstance(src, dict):
                src = _safe_text(src.get("name"))
            all_rows.append({
                "date": dt,
                "keyword": kw,
                "title": title,
                "description": desc,
                "source": src,
                "url": _safe_text(a.get("url")),
                "risk_score": _risk_score_from_text(title, desc),
            })
    return prepare_news_history(pd.DataFrame(all_rows))

def build_daily_news_features(df_news, analysis_date):
    idx = pd.date_range(analysis_date - timedelta(days=LOOKBACK_DAYS), analysis_date, freq="D", tz="UTC")
    full = pd.DataFrame({"date": idx})
    if df_news.empty:
        empty = full.copy()
        empty["news_count"] = 0
        empty["source_count"] = 0
        empty["keyword_count"] = 0
        empty["avg_risk"] = 0.0
        empty["max_risk"] = 0.0
        empty["has_news"] = 0
        empty["news_count_7d"] = 0.0
        empty["avg_risk_7d"] = 0.0
        for kw in ["oil", "war", "sanctions", "china", "taiwan", "inflation"]:
            empty[f"kw_{kw}"] = 0.0
        for theme in EVENT_TOPICS:
            empty[f"theme_{theme.lower()}"] = 0.0
        return empty
    base = prepare_news_history(df_news)
    base["date"] = pd.to_datetime(base["date"], utc=True).dt.floor("D")
    base["title_desc"] = (base["title"].fillna("") + " " + base["description"].fillna("")).str.lower()
    daily = base.groupby("date").agg(news_count=("title", "count"), source_count=("source", "nunique"), keyword_count=("keyword", "nunique"), avg_risk=("risk_score", "mean"), max_risk=("risk_score", "max")).reset_index()
    daily["has_news"] = (daily["news_count"] > 0).astype(int)
    for kw in ["oil", "war", "sanctions", "china", "taiwan", "inflation"]:
        tmp = base.assign(hit=base["title_desc"].str.contains(kw, regex=False).astype(int)).groupby("date")["hit"].sum().reset_index().rename(columns={"hit": f"kw_{kw}"})
        daily = daily.merge(tmp, on="date", how="left")
    for theme, kws in EVENT_TOPICS.items():
        tmp = base.assign(hit=base["title_desc"].apply(lambda text: sum(1 for kw in kws if kw in text))).groupby("date")["hit"].sum().reset_index().rename(columns={"hit": f"theme_{theme.lower()}"})
        daily = daily.merge(tmp, on="date", how="left")
    daily = full.merge(daily, on="date", how="left").fillna(0)
    daily["news_count_7d"] = daily["news_count"].rolling(7, min_periods=1).sum()
    daily["avg_risk_7d"] = daily["avg_risk"].rolling(7, min_periods=1).mean()
    return daily


# 5.1. 뉴스 수집 실행

- 이 셀을 실행하면 뉴스 API 호출, master 병합, snapshot 저장, 일별 뉴스 피처 생성까지 수행합니다.


In [6]:
from_dt = ANALYSIS_DATE - timedelta(days=NEWS_COLLECTION_LOOKBACK_DAYS)
to_dt = ANALYSIS_DATE + timedelta(days=1)
latest_news_before = load_latest_snapshot_df(NEWS_SNAPSHOT_DIR, "news_history", empty_columns=NEWS_COLUMNS)
master_news_before = load_history_csv(NEWS_MASTER_PATH, empty_columns=NEWS_COLUMNS)
print(f"Latest snapshot rows(before): {len(latest_news_before)}")
print(f"Master history rows(before): {len(master_news_before)}")
print(f"Configured news collection lookback days: {NEWS_COLLECTION_LOOKBACK_DAYS}")
new_news = fetch_news_from_api(EVENT_KEYWORDS, from_dt, to_dt, enabled=ENABLE_NEWS_API_FETCH)
print(f"Newly fetched news rows: {len(new_news)}")
master_news, master_news_changed = upsert_history_file(NEWS_MASTER_PATH, new_news, prepare_fn=prepare_news_history, sort_cols=["date", "article_id"], dedupe_cols=["article_id"])
window_news = master_news.copy()
if len(window_news) > 0:
    window_news = window_news[window_news["date"] >= from_dt].copy()
news_path, news_written = save_snapshot_if_changed(window_news, NEWS_SNAPSHOT_DIR, "news_history", RUN_TS)
df_news = prepare_news_history(load_history_csv(NEWS_MASTER_PATH, empty_columns=NEWS_COLUMNS))
if len(df_news) > 0:
    df_news = df_news[df_news["date"] >= (ANALYSIS_DATE - timedelta(days=LOOKBACK_DAYS))].copy()
daily_news = build_daily_news_features(df_news, ANALYSIS_DATE)
print(f"Master news path: {NEWS_MASTER_PATH}")
print(f"Master news updated: {master_news_changed}")
print(f"News snapshot path: {news_path}")
print(f"News snapshot written: {news_written}")
print(f"News rows in lookback window: {len(df_news)}")
print(f"Daily news feature rows: {len(daily_news)}")
if len(df_news) > 0:
    print(f"News date range: {df_news['date'].min()} -> {df_news['date'].max()}")
    print(df_news[["date", "keyword", "source", "title"]].tail(10).to_string(index=False))
else:
    print("No news data available after collection.")


Latest snapshot rows(before): 2331
Master history rows(before): 2353
Configured news collection lookback days: 30
API key status -> GNEWS: True (len=32), NEWSAPI: False (len=0)
Requested news window: 2026-02-25 -> 2026-03-28
Effective GNews window: 2026-02-26 -> 2026-03-28 (free-plan safe)
Effective GNews max articles/query: 10
GNews keyword='red sea shipping' -> totalArticles=62, fetched=10
GNews throttle: waiting 2.8s before keyword='shipping disruption'
GNews keyword='shipping disruption' -> totalArticles=238, fetched=10
GNews throttle: waiting 2.6s before keyword='oil supply disruption'
GNews keyword='oil supply disruption' -> totalArticles=450, fetched=10
GNews throttle: waiting 3.0s before keyword='sanctions risk'
GNews keyword='sanctions risk' -> totalArticles=18, fetched=10
GNews throttle: waiting 3.1s before keyword='middle east conflict'
GNews keyword='middle east conflict' -> totalArticles=8827, fetched=10
GNews throttle: waiting 2.9s before keyword='tanker attacks'
GNews ke

# 6. 가격 데이터 수집


In [7]:
def _flatten_close(df):
    if df is None or df.empty:
        return pd.DataFrame(columns=["date", "close", "volume"])
    x = df.copy()
    if isinstance(x.columns, pd.MultiIndex):
        x.columns = ["_".join([str(p) for p in col if p != ""]) if isinstance(col, tuple) else str(col) for col in x.columns]
    close_cols = [col for col in x.columns if str(col).lower().startswith("close")]
    volume_cols = [col for col in x.columns if str(col).lower().startswith("volume")]
    close_series = pd.to_numeric(x[close_cols[0]], errors="coerce") if close_cols else pd.Series(np.nan, index=x.index)
    volume_series = pd.to_numeric(x[volume_cols[0]], errors="coerce") if volume_cols else pd.Series(np.nan, index=x.index)
    out = pd.DataFrame({"date": x.index, "close": close_series.values, "volume": volume_series.values}).reset_index(drop=True)
    out["date"] = pd.to_datetime(out["date"], utc=True, errors="coerce")
    out["close"] = pd.to_numeric(out["close"], errors="coerce")
    out["volume"] = pd.to_numeric(out["volume"], errors="coerce")
    return out.dropna(subset=["date", "close"])

def fetch_symbol_history(symbol, start_date, end_date, enabled=False):
    if not enabled:
        return pd.DataFrame(columns=["date", "close", "volume"])
    try:
        df = yf.download(symbol, start=start_date, end=end_date, progress=False, auto_adjust=True, threads=False)
        flat = _flatten_close(df)
        if flat.empty:
            print(f"[WARNING] Price download returned no usable rows for symbol={symbol}")
        return flat
    except Exception as e:
        print(f"[WARNING] Price download failed for symbol={symbol}: {e}")
        return pd.DataFrame(columns=["date", "close", "volume"])

def build_asset_map():
    rows = []
    for name, sym in COMMODITIES.items():
        rows.append({"symbol": sym, "asset_name": name, "category": name, "kind": "commodity"})
    for cat, syms in STOCKS.items():
        for s in syms:
            rows.append({"symbol": s, "asset_name": ASSET_LABELS.get(s, s), "category": cat, "kind": "stock"})
    return pd.DataFrame(rows)

asset_map = build_asset_map()
start_date = ANALYSIS_DATE - timedelta(days=LOOKBACK_DAYS)
end_date = ANALYSIS_DATE + timedelta(days=1)
fresh_rows = []
if ENABLE_PRICE_REFRESH:
    for _, a in asset_map.iterrows():
        raw = fetch_symbol_history(a["symbol"], start_date, end_date, enabled=True)
        if raw.empty:
            continue
        raw["symbol"] = a["symbol"]
        raw["asset_name"] = a["asset_name"]
        raw["category"] = a["category"]
        raw["kind"] = a["kind"]
        fresh_rows.append(raw)
else:
    print("ENABLE_PRICE_REFRESH=False -> skipping live price refresh.")
fresh_prices = pd.concat(fresh_rows, ignore_index=True) if fresh_rows else pd.DataFrame(columns=PRICE_COLUMNS)
latest_price_before = load_latest_snapshot_df(PRICE_SNAPSHOT_DIR, "price_history", empty_columns=PRICE_COLUMNS)
if len(latest_price_before) > 0:
    latest_price_before = latest_price_before[latest_price_before["symbol"].isin(CURRENT_PRICE_SYMBOLS)].copy()
master_price_before = load_history_csv(PRICE_MASTER_PATH, empty_columns=PRICE_COLUMNS)
if len(master_price_before) > 0:
    master_price_before = master_price_before[master_price_before["symbol"].isin(CURRENT_PRICE_SYMBOLS)].copy()
df_prices = merge_history(master_price_before if len(master_price_before) > 0 else latest_price_before, fresh_prices, prepare_fn=prepare_price_history, sort_cols=["symbol", "date"], dedupe_cols=["date", "symbol"])
if not df_prices.empty:
    df_prices = df_prices.sort_values(["symbol", "date"]).reset_index(drop=True)
    def _safe_change(series, periods=1):
        s = pd.to_numeric(series, errors="coerce")
        base = s.shift(periods).replace(0, np.nan)
        return s.div(base) - 1
    g = df_prices.groupby("symbol", group_keys=False)
    df_prices["return_1d"] = g["close"].transform(lambda s: _safe_change(s, 1))
    df_prices["vol_20d"] = g["return_1d"].rolling(20).std().reset_index(level=0, drop=True)
    df_prices["mom_5d"] = g["close"].transform(lambda s: _safe_change(s, 5))
    df_prices["mom_20d"] = g["close"].transform(lambda s: _safe_change(s, 20))
    df_prices["volume_chg_5d"] = g["volume"].transform(lambda s: _safe_change(s, 5))
    df_prices = prepare_price_history(df_prices)
    df_prices = df_prices[df_prices["date"] >= start_date].sort_values(["symbol", "date"]).reset_index(drop=True)
master_prices, master_price_changed = upsert_history_file(PRICE_MASTER_PATH, df_prices, prepare_fn=prepare_price_history, sort_cols=["symbol", "date"], dedupe_cols=["date", "symbol"])
if len(master_prices) > 0:
    filtered_master_prices = master_prices[master_prices["symbol"].isin(CURRENT_PRICE_SYMBOLS)].copy()
    if len(filtered_master_prices) != len(master_prices):
        master_prices = prepare_price_history(filtered_master_prices)
        master_prices.to_csv(PRICE_MASTER_PATH, index=False, encoding="utf-8")
        master_price_changed = True
window_prices = master_prices.copy()
if not window_prices.empty:
    window_prices = window_prices[window_prices["date"] >= start_date].copy()
price_path, price_written = save_snapshot_if_changed(window_prices, PRICE_SNAPSHOT_DIR, "price_history", RUN_TS)
df_prices = prepare_price_history(load_history_csv(PRICE_MASTER_PATH, empty_columns=PRICE_COLUMNS))
if len(df_prices) > 0:
    df_prices = df_prices[df_prices["symbol"].isin(CURRENT_PRICE_SYMBOLS)].copy()
    df_prices = df_prices[df_prices["date"] >= start_date].copy()
print(f"Master price path: {PRICE_MASTER_PATH}")
print(f"Master price updated: {master_price_changed}")
print(f"Price snapshot path: {price_path}")
print(f"Price snapshot written: {price_written}")
print(f"Price rows (lookback window): {len(df_prices)}")
print(f"Tracked symbols: {df_prices['symbol'].nunique() if len(df_prices)>0 else 0}")
if df_prices.empty:
    print("No price data available.")
else:
    latest_snapshot = (
        df_prices.sort_values("date")
        .groupby("symbol")
        .tail(1)[["symbol", "category", "close", "return_1d", "vol_20d"]]
        .sort_values("category")
    )
    print(latest_snapshot.head(15))
    print(f"Date range: {df_prices['date'].min()} -> {df_prices['date'].max()}")
latest_price_file = _latest_snapshot_path(PRICE_SNAPSHOT_DIR, "price_history")
if latest_price_file and latest_price_file.exists():
    st = latest_price_file.stat()
    print(f"latest price file: {latest_price_file}")
    print(f"last_modified: {datetime.fromtimestamp(st.st_mtime)}")
    print(f"size_bytes: {st.st_size}")


Master price path: c:\Users\user\Documents\GitHub\GB_practicing\Stock_analysis\data\event_driven_stock_predictor\price_master_history.csv
Master price updated: True
Price snapshot path: c:\Users\user\Documents\GitHub\GB_practicing\Stock_analysis\data\event_driven_stock_predictor\price_snapshots\price_history_20260327_014947.csv
Price snapshot written: True
Price rows (lookback window): 12024
Tracked symbols: 24
      symbol           category       close  return_1d   vol_20d
1511     DAL           Airlines   66.860001  -0.016620  0.031316
5039    JETS           Airlines   25.010000  -0.011462  0.022546
11087    XLF              Banks   49.049999  -0.005878  0.007636
5543     JPM              Banks  291.660004  -0.012728  0.010021
1007     BTU               Coal   37.430000  -0.006635  0.040195
6047     LMT  Defense_Aerospace  627.330017   0.005014  0.016291
4031     ITA  Defense_Aerospace  220.119995  -0.025414  0.015882
7559     NEM        Gold_Miners   99.360001  -0.021277  0.029722


# 7. 데이터 커버리지 점검


In [8]:
print("Data coverage report")
print("=" * 90)

if 'df_news' in globals() and isinstance(df_news, pd.DataFrame) and len(df_news) > 0:
    news_dates = pd.to_datetime(df_news['date'], utc=True, errors='coerce').dropna()
    print("[News]")
    print(f"Rows: {len(df_news):,}")
    print(f"Date range: {news_dates.min()} -> {news_dates.max()}")
    print(f"Unique sources: {df_news['source'].nunique() if 'source' in df_news.columns else 0}")
else:
    print("[News]")
    print("No news data available.")

print("\n" + "-" * 90)

if 'df_prices' in globals() and isinstance(df_prices, pd.DataFrame) and len(df_prices) > 0:
    ptmp = df_prices.copy()
    ptmp['date'] = pd.to_datetime(ptmp['date'], utc=True, errors='coerce')
    ptmp = ptmp.dropna(subset=['date'])

    summary = (
        ptmp.groupby(['symbol', 'category'], as_index=False)
        .agg(
            rows=('date', 'count'),
            first_date=('date', 'min'),
            last_date=('date', 'max'),
            last_close=('close', 'last')
        )
        .sort_values(['category', 'symbol'])
        .reset_index(drop=True)
    )
    summary['calendar_days'] = (summary['last_date'] - summary['first_date']).dt.days

    print("[Prices by symbol]")
    print(f"Total rows: {len(ptmp):,}")
    print(f"Tracked symbols: {summary['symbol'].nunique()}")
    print(f"Overall date range: {ptmp['date'].min()} -> {ptmp['date'].max()}")

    print("\nCoverage table (head 30):")
    print(summary[['symbol', 'category', 'rows', 'first_date', 'last_date', 'calendar_days', 'last_close']].head(30).to_string(index=False))

    print("\nCoverage stats:")
    print(summary['rows'].describe().round(1).to_string())

    low_cov = summary[summary['rows'] < 200]
    if len(low_cov) > 0:
        print("\nSymbols with low coverage (<200 rows):")
        print(low_cov[['symbol', 'category', 'rows', 'first_date', 'last_date']].to_string(index=False))
else:
    print("[Prices]")
    print("No price data available.")

print("=" * 90)


Data coverage report
[News]
Rows: 2,574
Date range: 2026-02-10 15:59:00+00:00 -> 2026-03-26 13:50:24+00:00
Unique sources: 297

------------------------------------------------------------------------------------------
[Prices by symbol]
Total rows: 12,024
Tracked symbols: 24
Overall date range: 2024-03-27 00:00:00+00:00 -> 2026-03-26 00:00:00+00:00

Coverage table (head 30):
symbol          category  rows                first_date                 last_date  calendar_days  last_close
   DAL          Airlines   501 2024-03-27 00:00:00+00:00 2026-03-26 00:00:00+00:00            729   66.860001
  JETS          Airlines   501 2024-03-27 00:00:00+00:00 2026-03-26 00:00:00+00:00            729   25.010000
   JPM             Banks   501 2024-03-27 00:00:00+00:00 2026-03-26 00:00:00+00:00            729  291.660004
   XLF             Banks   501 2024-03-27 00:00:00+00:00 2026-03-26 00:00:00+00:00            729   49.049999
   BTU              Coal   501 2024-03-27 00:00:00+00:00 2026-03-26 00:

# 8. 최근 이슈 모니터


In [9]:
print("Recent issue monitor")
print("=" * 90)

if 'df_news' not in globals() or not isinstance(df_news, pd.DataFrame) or len(df_news) == 0:
    print("No news data available.")
else:
    recent_cutoff = ANALYSIS_DATE - timedelta(days=30)
    monitor = prepare_news_history(df_news).copy()
    monitor['date'] = pd.to_datetime(monitor['date'], utc=True, errors='coerce')
    monitor = monitor.dropna(subset=['date'])
    monitor = monitor[monitor['date'] >= recent_cutoff].copy()

    if monitor.empty:
        print(f"No news rows in the last 30 days since {recent_cutoff:%Y-%m-%d}.")
    else:
        monitor['text'] = (monitor['title'].fillna('') + ' ' + monitor['description'].fillna('')).str.lower()
        rows = []
        for theme, kws in EVENT_TOPICS.items():
            hit_counts = monitor['text'].apply(lambda text: sum(1 for kw in kws if kw in text))
            matched = monitor[hit_counts > 0].copy()
            article_hits = int((hit_counts > 0).sum())
            keyword_hits = int(hit_counts.sum())
            avg_risk = float(matched['risk_score'].mean()) if len(matched) > 0 else 0.0
            issue_score = keyword_hits * (1.0 + avg_risk)
            top_keywords = []
            if len(matched) > 0:
                kw_counts = {kw: int(matched['text'].str.contains(re.escape(kw), regex=True).sum()) for kw in kws}
                top_keywords = [kw for kw, cnt in sorted(kw_counts.items(), key=lambda x: (-x[1], x[0])) if cnt > 0][:3]
            rows.append({
                'Theme': theme,
                'Theme_KR': {
                    'Geopolitical_Conflict': '지정학 분쟁',
                    'Monetary_Inflation': '통화정책/인플레이션',
                    'Tariff_Trade_Policy': '관세/무역정책',
                    'China_Demand_Stimulus': '중국 수요/부양책',
                    'Energy_Oil_Gas': '에너지/원유/가스',
                    'Shipping_Logistics': '해운/물류',
                    'Semiconductor_AI': '반도체/AI',
                    'Defense_Aerospace': '방산/항공우주',
                    'Financial_Stress': '금융불안',
                    'Precious_Metals_Safe_Haven': '금/안전자산',
                    'Battery_Materials': '배터리 소재',
                    'Earnings_Guidance': '실적/가이던스',
                }.get(theme, theme),
                'Article_Count_30D': article_hits,
                'Keyword_Hits_30D': keyword_hits,
                'Avg_Risk_30D': round(avg_risk, 3),
                'Issue_Score_30D': round(issue_score, 2),
                'Top_Keywords': ', '.join(top_keywords) if top_keywords else '-',
            })

        issue_df = pd.DataFrame(rows).sort_values(['Issue_Score_30D', 'Keyword_Hits_30D', 'Article_Count_30D'], ascending=[False, False, False]).reset_index(drop=True)
        print(f"Recent news rows used: {len(monitor):,} (window: {recent_cutoff:%Y-%m-%d} -> {ANALYSIS_DATE:%Y-%m-%d})")
        print("\n[Top 5 issues]")
        print(issue_df[['Theme_KR', 'Article_Count_30D', 'Keyword_Hits_30D', 'Avg_Risk_30D', 'Issue_Score_30D', 'Top_Keywords']].head(5).to_string(index=False))
        print("\n[Bottom 5 issues]")
        print(issue_df[['Theme_KR', 'Article_Count_30D', 'Keyword_Hits_30D', 'Avg_Risk_30D', 'Issue_Score_30D', 'Top_Keywords']].tail(5).sort_values(['Issue_Score_30D', 'Keyword_Hits_30D', 'Article_Count_30D'], ascending=[True, True, True]).to_string(index=False))

print("=" * 90)


Recent issue monitor
Recent news rows used: 2,545 (window: 2026-02-25 -> 2026-03-27)

[Top 5 issues]
  Theme_KR  Article_Count_30D  Keyword_Hits_30D  Avg_Risk_30D  Issue_Score_30D                                                Top_Keywords
 에너지/원유/가스                327               358         0.088           389.53                         oil prices, brent crude, lng supply
    지정학 분쟁                134               135         0.125           151.86 middle east conflict, defense spending, shipping disruption
통화정책/인플레이션                112               118         0.030           121.58              treasury yields, dollar strength, fed rate cut
   실적/가이던스                 58                58         0.031            59.80               margin pressure, earnings beat, earnings miss
   관세/무역정책                 50                50         0.021            51.07                 tariff hike, export controls, import duties

[Bottom 5 issues]
 Theme_KR  Article_Count_30D  Keyword_Hits_30D